# 2. Ingest – Catálogo de sucursales (`dealers.csv`)
**Capa:** raw ➜ bronze  |  **Destino:** `vehiculos.bronze.dealers`

- Se compara la lectura con `inferSchema` vs. **schema explícito**: con `inferSchema` el código postal `06457` se convierte en número y pierde el cero inicial, por eso se define el schema.
- Estandarización de textos (`trim`, mayúsculas en el código de estado).
- Columnas de auditoría: `source_file` e `ingestion_date`.

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "vehiculos")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsjgl99")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/dealers.csv"

### Lectura con `inferSchema` (solo exploración)

In [0]:
df_dealers_infer = spark.read.option('header', True)\
                             .option('inferSchema', True)\
                             .csv(ruta)

df_dealers_infer.printSchema()
display(df_dealers_infer)

### Lectura con schema explícito

In [0]:
dealers_schema = StructType(fields=[StructField("dealer_no", StringType(), False),
                                    StructField("zip_code", StringType(), True),
                                    StructField("city", StringType(), True),
                                    StructField("state_code", StringType(), True),
                                    StructField("state_name", StringType(), True),
                                    StructField("latitude", DoubleType(), True),
                                    StructField("longitude", DoubleType(), True)
])

In [0]:
df_dealers = spark.read\
.option('header', True)\
.schema(dealers_schema)\
.csv(ruta)

df_dealers.printSchema()

### Estandarización y columnas de auditoría

In [0]:
dealers_final_df = df_dealers.select(
    trim(col("dealer_no")).alias("dealer_no"),
    lpad(trim(col("zip_code")), 5, "0").alias("zip_code"),
    initcap(trim(col("city"))).alias("city"),
    upper(trim(col("state_code"))).alias("state_code"),
    initcap(trim(col("state_name"))).alias("state_name"),
    col("latitude"),
    col("longitude"),
    col("_metadata.file_name").alias("source_file")
).withColumn("ingestion_date", current_timestamp())

### Validación previa a la carga

In [0]:
total = dealers_final_df.count()
duplicados = total - dealers_final_df.select("dealer_no").distinct().count()
coordenadas_invalidas = dealers_final_df.filter(
    ~col("latitude").between(-90, 90) | ~col("longitude").between(-180, 180)
).count()

print(f"Sucursales leídas    : {total}")
print(f"dealer_no duplicados : {duplicados}")
print(f"Coordenadas inválidas: {coordenadas_invalidas}")

### Escritura en bronze

In [0]:
columnas_tabla = spark.table(f"{catalogo}.{esquema}.dealers").columns

dealers_final_df.select(*columnas_tabla) \
    .write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.dealers")

In [0]:
display(spark.table(f"{catalogo}.{esquema}.dealers"))